In [ ]:
# APPENDIX 1
import random
from collections import deque
from collections import defaultdict

# Student Identity 
LAST_NAME = "Bacor" # Replace with surname 
STUDENT_ID = "TUPM-26-1879" 
FAVORITE_ARTIST = "IVOS"
SEED_NUM = int(STUDENT_ID[-1])
WIDTH = 100

# Define sources and categories
sources = [f"{LAST_NAME}_T{i}" for i in range (1, 6)]
categories = ["TEMPERATURE", "VOLTAGE", "CURRENT", "PRESSURE", "SPEED"]

# Make a list containing the indexes of the FAVORITE_ARTIST parameter.
# c.ascii() check to ensure each character is a valid ascii char.
char_pos = [x for x, c in enumerate(FAVORITE_ARTIST) if c.isascii()]
data_queue = deque()
def generate_values(category, char_idx):
    # Get the position in the FAVORITE_ARTIST parameters
    char = FAVORITE_ARTIST[char_idx]

    if category == "TEMPERATURE":
        return (round(SEED_NUM + ord(char) + 1, 2))
    elif category == "VOLTAGE":
        return (round(SEED_NUM + ord(char) + 3, 2))
    elif category == "CURRENT":
        return (round(SEED_NUM + ord(char) / 2, 2))
    elif category == "PRESSURE":
        return (round(SEED_NUM + ord(char) - 1, 2))
    else:
        return (round(SEED_NUM + ord(char) / 3, 2))
                
for source in sources:
    for cat in categories:
        c_idx = random.choice(char_pos)
        val = generate_values(cat, c_idx)
        data_queue.append({
            "source": source,
            "category": cat,
            "value": val
        })
        

# ADDING REPEATED AND EXTRA RECORDS
repeated_record = data_queue[0].copy()
data_queue.append(repeated_record)

extra_val_1 = generate_values("CURRENT", 3)
extra_val_2 = generate_values("PRESSURE", 1)

data_queue.append({
    "source": sources[2],
    "category": categories[2],
    "value": extra_val_1
})

data_queue.append({
    "source": sources[1],
    "category": categories[3],
    "value": extra_val_2
})


# IDENTIFICATION OF REPEATED AND DISTINCT DATA
seen_records = []
distinct_records = []
duplicate_records = []

for r in data_queue:
    # Convert into tuple to make comparison reliable.
    record_tuple = (r["source"], r["category"], r["value"])

    if record_tuple in seen_records:
        if record_tuple not in duplicate_records:
            duplicate_records.append(record_tuple)
    else: 
        seen_records.append(record_tuple)
        distinct_records.append(r)

print("=" * WIDTH)
print(f"{'REPEATED / DISTINCT DATA IDENTIFICATION':^{WIDTH}}")
print("-" * WIDTH)
print(f"Distinct Data Count: {len(distinct_records)}")
for i in duplicate_records:
    print(
        f"DUPLICATE FOUND IN Source: {i[0]}, Category: {i[1]} with a value of {i[2]}"
    )
print("=" * WIDTH)

# PROCESSING DATA ABNORMALITIES
## Removing duplicates
unique_raw_records = []
for r in data_queue:
    if r not in unique_raw_records:
        unique_raw_records.append(r)

# ORGANIZATION OF DATA
grouped_measurements = defaultdict(list)
for r in unique_raw_records:
    # Convert first into tuple to ensure reliability.
    key = (r["source"], r["category"])
    grouped_measurements[key].append(r["value"])

## Average the Multiple Values
processed_records = []
for (src, cat), values in grouped_measurements.items():
    # Calculate the average of all measurements for the specific source and category
    averaged_values = round(sum(values) / len(values), 2)

    processed_records.append({
        "source": src,
        "category": cat,
        "value": averaged_values
    })

# NUMERICAL AND CATEGORICAL SUMMARIES
category_summaries= {}
for cat in categories:
    # Create a list of values in processed_records in respective categories.
    cat_val = [r["value"] for r in processed_records if r["category"] == cat]

    if cat_val:
        cat_min = min(cat_val)
        cat_max = max(cat_val)
        cat_avg = round(sum(cat_val) / len(cat_val), 2)
        cat_count = len(cat_val)
    else:
        cat_min, cat_max, cat_avg, cat_count = 0, 0, 0, 0

    category_summaries[cat] = {
        "min": cat_min,
        "max": cat_max,
        "avg": cat_avg,
        "count": cat_count
    }

header = (
    f"{'Category':<15} | {'Count':<6} | {'Min':<10} | {'Max':<10} | {'Average':<10}"
)
width = len(header)
print("=" * width)
print(f"{'NUMERICAL AND CATEGORICAL SUMMARIES':^{width}}")
print("=" * width)
print(header)
print("-" * width)
for cat, stats in category_summaries.items():
    print(
        f"{cat:<15} | {stats["count"]:<6} | {stats["min"]:<10} | {stats["max"]:<10} | {stats["avg"]:<10}"
    )
highest_data = max(distinct_records, key=lambda x: x["value"])
lowest_data = min(distinct_records, key=lambda x: x["value"])
print("-" * width)
print(f"Highest Data Recorded: Source in {highest_data['source']}, Category: {highest_data['category']} with a Value {highest_data['value']}")
print(f"Lowest Data Recorded: Source in {lowest_data['source']}, Category: {lowest_data['category']} with a Value {lowest_data['value']}")
print("=" * width)

# PROCESSED RESULTS ARRANGEMENT
ordered_records = sorted(processed_records, key=lambda x: (x["category"], x["value"]))

# PRESENT ORIGINAL AND PROCESSED RESULTS
print("=" * 100)
print(f"{'GENERATED DATA SET':^15}")
print("=" * 100)
print("-" * 100)
for r in data_queue:
    print(f"From Source: {r["source"]}, Category: {r["category"]}, Value {r["value"]}")
print("=" * 100)

print("=" * 100)
print(f"{'ORGANZED DATA SET':^15}")
print("=" * 100)
print("-" * 100)
for (src, cat), val in grouped_measurements.items():
    print(f"Source: {src} | Category: {cat:<20} | Value {str(",".join(map(str, val)))}")
print("=" * 100)

orig_matrix = {s: {c: [] for c in categories} for s in sources}
for r in data_queue:
    orig_matrix[r["source"]][r["category"]].append(r["value"])

header = (
    f"{'Category':<15} | {'Value':<10} | {'Source':<10}"
)
width = len(header)
print("=" * width)
print(f"{'ARRANGED PROCESSED RESULTS BY Category, Value':^{width}}")
print("=" * width)
print(header)
print("-" * width)
for r in ordered_records:
    print(
        f"{r["category"]:<15} | {r["value"]:<10} | {r["source"]:<10}"
    )
print("=" * width)

header = (
    f"{'SOURCE ID':<15} | " + " |".join([f"{c[:4]:<25}" for c in categories])
)
width = len(header)
print("=" * width)
print(f"{'ORIGINAL DATA':^{width}}")
print("=" * width)
print(header)
print("-" * width)

for src in sources:
    row_cells = []
    for cat in categories:
        vals = orig_matrix[src][cat]
        if len(vals) == 1:
            val_str = f"{vals[0]:.2f}"
        else:
            val_str = str(vals) # display more than 1 values
        row_cells.append(f"{val_str:<25}")
    print(f"{src:<15} | " + " |".join(row_cells))
print("=" * width)

# FINAL PRINTING
print("=" * 100)
print(f"{'CONSOLIDATED SUMMARY REPORT':^100}")
print("-" * 100)
print(f"Total Data Count: {len(data_queue)}")
print(f"Total Test Sources: {len(sources)}")
print(f"Total Categories: {len(categories)}")
print("=" * 100)

proc_matrix = {s: {c: None for c in categories} for s in sources}
for r in processed_records:
    proc_matrix[r["source"]][r["category"]] = (r["value"])
header = (
    f"{'SOURCE ID':<15} | " + " |".join([f"{c[:4]:<25}" for c in categories])
)
width = len(header)
print("=" * width)
print(f"{'FINAL ORDERED DATA SET':^{width}}")
print("=" * width)
print(header)
print("-" * width)

for src in sources:
    row_cells = []
    for cat in categories:
        vals = orig_matrix[src][cat]
        if vals is not None:
            val_str = f"{vals[0]:.2f}"
        else:
            val_str = "N/A" # display more than 1 values
        row_cells.append(f"{val_str:<25}")
    print(f"{src:<15} | " + " |".join(row_cells))
print("=" * width)




                              REPEATED / DISTINCT DATA IDENTIFICATION                               
----------------------------------------------------------------------------------------------------
Distinct Data Count: 27
DUPLICATE FOUND IN Source: Bacor_T1, Category: TEMPERATURE with a value of 89
              NUMERICAL AND CATEGORICAL SUMMARIES              
Category        | Count  | Min        | Max        | Average   
---------------------------------------------------------------
TEMPERATURE     | 5      | 83.0       | 96.0       | 90.0      
VOLTAGE         | 5      | 85.0       | 98.0       | 93.6      
CURRENT         | 5      | 48.0       | 52.0       | 49.5      
PRESSURE        | 5      | 87.0       | 94.0       | 91.7      
SPEED           | 5      | 33.33      | 37.67      | 34.6      
Highest Data Recorded: Source in Bacor_T4, Category: VOLTAGE with a Value 98
Lowest Data Recorded: Source in Bacor_T1, Category: SPEED with a Value 33.33
GENERATED DATA SET
-----------

In [117]:
students = [f"{LAST_NAME}_S{i}" for i in range(1, 6)]
subjects = ["PROGRAMMING", "MATHEMATICS", "ELECTRONICS", "PHYSICS", "ENGLISH"]
PASSING_SCORE = 75

char_pos = [x for x, c in enumerate(FAVORITE_ARTIST) if c.isascii()]
score_queue = deque()
def generate_scores(subjects, char_idx):
    # Get the position in the FAVORITE_ARTIST parameters
    char = FAVORITE_ARTIST[char_idx]

    if subjects == "PROGRAMMING":
        return (round(SEED_NUM + ord(char) + 1, 2))
    elif subjects == "MATHEMATICS":
        return (round(SEED_NUM + ord(char) + 3, 2))
    elif subjects == "ELECTRONICS":
        return (round(SEED_NUM + ord(char), 2))
    elif subjects == "PHYSICS":
        return (round(SEED_NUM + ord(char) - 10, 2))
    else:
        return (round(SEED_NUM + ord(char) - 5, 2))
                
for stud in students:
    for sub in subjects:
        c_idx = random.choice(char_pos)
        scr = generate_scores(sub, c_idx)
        score_queue.append({
            "student": stud,
            "subject": sub,
            "score": scr
        })

# CREATION OF REPEATED AND MISSING SCORES
repeated_score = score_queue[0].copy()
score_queue.append(repeated_score)
score_queue.append({
    "student": students[2],
    "subject": "PROGRAMMING",
    "score": None
})

organized_scores = defaultdict(lambda: {sub: [] for sub in subjects})
for r in score_queue:
    # Convert first into tuple to ensure reliability.
    organized_scores[r["student"]][r["subject"]].append(r["score"])

# REPEATED / MISSING SCORE IDENTIFICATION
seen_scores = set()
distinct_scores = []
duplicate_scores = []
missing_scores = []
cleaned_scores = {s: {sub: [] for sub in subjects} for s in students}

# IDENTIFY MISSING, DISTINCT AND DUPLICATE SCORES
for stud, sub_dict in organized_scores.items():
    for sub, scores in sub_dict.items():
        for scr in scores:
            if scr is None:
                missing_scores.append({
                    "student": stud,
                    "subject": sub,
                    "score": "N/A"
                })
                continue
            score_tuple = (stud, sub, scr)

            if score_tuple in seen_scores:
                duplicate_scores.append(score_tuple)
            else:
                seen_scores.add(score_tuple)
                distinct_scores.append({
                    "student": stud,
                    "subject": sub,
                    "score": scr
                })
                cleaned_scores[stud][sub].append(scr)

# PROCESSING SCORES
processed_scores = []
for stud in students:
    for sub in subjects:
        valid_scores = cleaned_scores[stud][sub]

        if valid_scores:
            averaged_values = round(sum(valid_scores) / len(valid_scores), 2)
        else:
            averaged_values = None

        processed_scores.append({
            "student": stud,
            "subject": sub,
            "score": averaged_values
        })

# Ensure that None is handled out.
valid_scores = [r for r in processed_scores if r["score"] is not None]

# PERFORMANCE EVALUATION
def evaluate_performance(scores):
    if scores is None:
        return "N/A"

    if scores >= 95:
        return "A"
    elif scores >= 85:
        return "B"
    elif scores >= 75:
        return "C"
    elif scores >= 65:
        return "D"
    else:
        return "F"

evaluation_list = []
for r in processed_scores:
    scores = r["score"]
    eval = evaluate_performance(scores)
    evaluation_list.append({
        "student": r["student"],
        "subject": r["subject"],
        "score": r["score"],
        "evaluation": eval
    })

print("=" * 100)
print(f"{'GENERATED SCORES':^15}")
print("=" * 100)
print("-" * 100)
for r in score_queue:
    stud = r["student"]
    sub = r["subject"]
    scr = r["score"]
    print(f"Student: {stud}, Subject: {sub}, Score {scr}")
print("=" * 100)

print("=" * 100)
print(f"{'ORGANZED SCORES':^100}")
print("=" * 100)
print("-" * 100)
for stud, sub_dict in organized_scores.items():
    for sub, scores in sub_dict.items():
        for scr in scores:
            print(f"Student: {stud} | Subject: {sub:<20} | Score: {scr}")
print("=" * 100)
print()

print("=" * 100)
print(f"{'REPEATED/MISSING DATA IDENTIFICATION':^15}")
print("-" * 100)
print(f"Distinct Data Count: {len(distinct_scores)}")

for i in duplicate_scores:
    print(f"DUPLICATE FOUND IN: {i[0]}, at Subject: {i[1]}, Score: {i[2]}")

for i in missing_scores:
        print(f"MISSING SCORE FOUND IN: {i["student"]}, at Subject: {i["subject"]}")
print("=" * 100)
print()

header = (
    f"{'Student':<12} | {'Subject':<15} | {'Score':<15} | {'Grade':<15}"
)
width = len(header)
print("=" * 100)
print(f"{'STUDENT SCORE EVALUATIONS':^15}")
print("=" * 100)
print(header)
print("-" * 100)
for r in evaluation_list:
    stud = r["student"]
    sub = r["subject"]
    scr = r["score"]
    eval = r["evaluation"]
    print(
        f"{stud:<12} | {sub:<15} | {scr:<15} | {eval:<15}"
    )
print("=" * 100)
print()

# STUDENT AND SUBJECT SUMMARIES
pass_counts = {sub: 0 for sub in subjects}
for r in processed_scores:
    if r["score"] is not None and r["score"] >= PASSING_SCORE:
        pass_counts[r["subject"]] += 1

max_passed = max(pass_counts.values())
top_subjects = [sub for sub, n in pass_counts.items() if n == max_passed]
highest_score = max(valid_scores, key=lambda x: x["score"])
lowest_score = min(valid_scores, key=lambda x: x["score"])
valid_scores = [r["score"] for r in valid_scores]
average_student_score = (round(sum(valid_scores) / len(valid_scores), 2) if valid_scores else None)
print("=" * 100)
print(f"{'STUDENT AND SUBJECT PERFORMANCE SUMMARIES':^100}")
print("-" * 100)
print(f"STUDENT WITH THE HIGHEST SCORE: {highest_score["student"]} at {highest_score["subject"]} with {highest_score["score"]}")
print(f"STUDENT WITH THE LOWEST SCORE: {lowest_score["student"]} at {lowest_score["subject"]} with {lowest_score["score"]}")
print(f"Average Student Score: {average_student_score}")
print(f"Most Passed Subject: {' , '.join(top_subjects)} with {max_passed} students.")
print("=" * 100)
print()

header = (
    f"{'Student Names':<15} | " + " | ".join([f"{s[0:]:<15}" for s in subjects])
)
width = len(header)
print("=" * 100)
print(f"{'ORDERED PERFORMANCE DATA':^{width}}")
print("=" * 100)
print(header)
print("-" * 100)
student_avgs = {}
for stud in students:
    s_scores = [r["score"] for r in processed_scores if r["student"] == stud and r["score"] is not None] 
    student_avgs[stud] = sum(s_scores) / len(s_scores) if s_scores else None

ranked_students = sorted(
    students,
    key = lambda s: (student_avgs[s] is not None, student_avgs[s] or 0),
    reverse=True
)

for stud in ranked_students:
    row_cells = []
    for sub in subjects:
        vals = cleaned_scores[stud][sub]
        if len(vals) == 1 and vals[0] is not None:
            val_str = f"{vals[0]:.2f}"
        else: 
            val_str = str(vals)
        row_cells.append(f"{val_str:<15}")
    print(f"{stud:<15} | " + " | ".join(row_cells))
print("=" * 100)
print()

GENERATED SCORES
----------------------------------------------------------------------------------------------------
Student: Bacor_S1, Subject: PROGRAMMING, Score 83
Student: Bacor_S1, Subject: MATHEMATICS, Score 91
Student: Bacor_S1, Subject: ELECTRONICS, Score 88
Student: Bacor_S1, Subject: PHYSICS, Score 85
Student: Bacor_S1, Subject: ENGLISH, Score 87
Student: Bacor_S2, Subject: PROGRAMMING, Score 96
Student: Bacor_S2, Subject: MATHEMATICS, Score 95
Student: Bacor_S2, Subject: ELECTRONICS, Score 82
Student: Bacor_S2, Subject: PHYSICS, Score 82
Student: Bacor_S2, Subject: ENGLISH, Score 77
Student: Bacor_S3, Subject: PROGRAMMING, Score 96
Student: Bacor_S3, Subject: MATHEMATICS, Score 85
Student: Bacor_S3, Subject: ELECTRONICS, Score 88
Student: Bacor_S3, Subject: PHYSICS, Score 72
Student: Bacor_S3, Subject: ENGLISH, Score 83
Student: Bacor_S4, Subject: PROGRAMMING, Score 89
Student: Bacor_S4, Subject: MATHEMATICS, Score 85
Student: Bacor_S4, Subject: ELECTRONICS, Score 92
Studen

In [127]:
# APPENDIX 3
# Generate Products and Customer Names
num_cust = 6
customers = [f"{LAST_NAME}_C{i}" for i in range(1, num_cust)]
PRODUCTS = {
    "NOTEBOOK": 50.00,
    "PEN": 10.00,
    "USB": 70.00,
    "HEADPHONES": 100.00,
    "CALCULATOR": 1200.00
}
# Convert into list for future use.
product_list = list(PRODUCTS)

# Number of transactions


def generate_trxn(customer, product, quantity):
    amount = None if quantity is None else round(PRODUCTS[product] * quantity, 2)
    return {
        "customer": customer, 
        "product": product,
        "quantity": quantity,
        "amount": amount
    }

trxn_queue = deque()
for i, customer in enumerate(customers):
    # quantity and product_idx generated by ChatGPT
    quantity = ((i + SEED_NUM) % 5) + 1
    char_idx = char_pos[i % len(char_pos)]
    product_idx =  (SEED_NUM + char_idx + i) % len(product_list)
    trxn_queue.append(
        generate_trxn(
        customer,
        product_list[product_idx],
        quantity    
    ))

repeated_trxn = trxn_queue[0].copy()
trxn_queue.append(repeated_trxn)
trxn_queue.append(generate_trxn(customers[1], "USB", None))

organized_trxns = defaultdict(lambda: {prod: [] for prod in product_list})
for r in trxn_queue:
    cust = r["customer"]
    prod = r["product"]
    # Convert first into tuple to ensure reliability.
    organized_trxns[cust][prod].append(r)

# REPEATED / MISSING trxn IDENTIFICATION
seen_trxns = set()
recorded_duplicates = set()
distinct_trxns = []
prod_trxns = []
duplicate_trxns = []
missing_trxns = []
cleaned_trxns = {c: {prod: [] for prod in product_list} for c in customers}

# IDENTIFY MISSING, DISTINCT AND DUPLICATE trxns
for cust, prod_dict in organized_trxns.items():
    for prod, trxns in prod_dict.items():
        for t in trxns:
            if t["quantity"] is None:
                missing_trxns.append({
                    "customer": cust,
                    "product": prod,
                    "trxn": "N/A"
                })
                continue

            # trxn_tuple generated by: ChatGPT
            trxn_tuple = (
                t["customer"],
                t["product"],
                t["quantity"],
                t["amount"]
            )

            if trxn_tuple in seen_trxns:
                # recorded_duplicates generated by: ChatGPT
                if trxn_tuple not in recorded_duplicates:
                    duplicate_trxns.append(t)
                    recorded_duplicates.add(trxn_tuple)
            else:
                seen_trxns.add(trxn_tuple)
                distinct_trxns.append(t)
            prod_trxns.append(t)
            cleaned_trxns[cust][prod].append(t)

processed_trxns = []
for cust in customers:
    for prod in product_list:
        valid = cleaned_trxns[cust][prod]
        processed_trxns.append({
            "customer": cust, 
            "product": prod,
            "units": sum(t["quantity"] for t in valid),
            "total": round(sum(t["amount"] for t in valid), 2)
        })

# Create a list that includes every processed trxn with a unit above 0.
purchase_records = [r for r in processed_trxns if r["units"] > 0]
customer_totals = {
    # Add the amount spent by each customer
    c: round(sum(r["total"] for r in processed_trxns if r["customer"] == c), 2) for c in customers
}

product_totals = {
    p: {
        "units": sum(r["units"] for r in processed_trxns if r["product"] == p),
        "revenue": round(sum(r["total"] for r in processed_trxns if r["product"] == p), 2)
    }
    for p in product_list
}

print("=" * 100)
print(f"{'GENERATED TRANSACTION RECORDS':^15}")
print("=" * 100)
print("-" * 100)
for r in trxn_queue:
    cust = r["customer"]
    prod = r["product"]
    qty = r["quantity"]
    amount = r["amount"]
    print(f"Customer: {cust}, Product: {prod}, Quantity {qty}, Amount: {amount}")
print("=" * 100)

header = (
    f"{'Customer':<12} | {'Product':<12} | {'Qty':<15} | {'Unit Price':<15} "
)
width = len(header)
print("=" * width)
print(f"{'ORGANIZED TRANSACTION RECORDS':^{width}}")
print("=" * width)
print(header)
print("-" * width)
row_count = 0
for cust in customers:
    for prod in product_list:
        for t in organized_trxns[cust][prod]:
            qty = "N/A" if t["quantity"] is None else t["quantity"]
            print(
                f"{cust:<12} | {prod:<12} | {qty:<15} | {PRODUCTS[prod]:<15}"
            )
            row_count += 1
print("-" * width)
print(f"Total Records: {row_count}")
print("=" * width)
print()

# REPEATED / DISTINCT DATA IDENTIFICATION
print("=" * 100)
print(f"{'REPEATED/MISSING DATA IDENTIFICATION':^15}")
print("-" * 100)
print(f"Distinct Data Count: {len(distinct_trxns)}")

for i in duplicate_trxns:
    print(f"DUPLICATE FOUND IN: {i["customer"]} | {i["product"]} | {i["amount"]}")

for i in missing_trxns:
        print(f"MISSING TRANSACTION FOUND IN: {i["customer"]} | {i["product"]}")
print("=" * 100)
print()

# TRANSACTION VALUES
header = (
    f"{'Customer':<12} | {'Product':<12} | {'Qty':<15} | {'Unit Price':<15} | {'Total Amount':<15} "
)
width = len(header)
print("=" * width)
print(f"{'TRANSACTION VALUES':^{width}}")
print("=" * width)
print(header)
print("-" * width)

running_total = 0
for t in sorted(prod_trxns, key=lambda t: t["customer"]):
    running_total += t["amount"]
    print(
        f"{t["customer"]:<12} | {t['product']:<12} | {t['quantity']:<15} | {PRODUCTS[t["product"]]:<15} | {t['amount']:<15}"
    )
print("-" * width)
print(f"{'OVERALL TOTAL:':<43} | {running_total:>25}")
print("=" * width)
print()

# CUSTOMER SUMMARIES
customer_summary = {}
for cust in customers:
    cust_txns = [t for t in prod_trxns if t["customer"] == cust]
    spent_by_product = {
        p: sum(r["amount"] for r in cust_txns if r["product"] == p) for p in product_list
    }

    customer_summary[cust] = {
        "txns": len(cust_txns),
        "units": sum(t["quantity"] for t in cust_txns),
        "spent": customer_totals[cust],
        "top_product": max(spent_by_product, key=spent_by_product.get) if cust_txns else "N/A"
    }

header = (
    f"{'Customer':<12} | {'Txns':<12} | {'Units':<15} | {'Amount Spent':<15} | {'Top Product':<15}"
)
width = len(header)
print("=" * width)
print(f"{'CUSTOMER SUMMARIES':^{width}}")
print("=" * width)
print(header)
print("-" * width)
for cust, s in customer_summary.items():
    print(
        f"{cust:<12} | {s['txns']:<12} | {s['units']:<15} | {s['spent']:<15} | {s['top_product']:<15}"
    )
top_spend = max(customer_totals.values())
top_customers = [c for c, v in customer_totals.items() if v == top_spend]
print(f"Top Customers: {" , ".join(top_customers)} with {top_spend:.2f}")
print("=" * width)

# PRODUCT SUMMARIES
product_summary = {}
for prod in product_list:
    prod_txns = [t for t in prod_trxns if t["product"] == prod]
    spent_by_customer = {
        c: sum(r["amount"] for r in prod_txns if r["customer"] == c) for c in customers
    }

    revenue = product_totals[prod]["revenue"]
    product_summary[prod] = {
        "txns": len(prod_txns),
        "units": product_totals[prod]["units"],
        "revenue": revenue,
        "avg": round(revenue / len(prod_txns), 2) if prod_txns else 0,
        "top_buyer": max(spent_by_customer, key=spent_by_customer.get) if prod_txns else "N/A"
    }

header = (
    f"{'Product':<12} | {'Txns':<12} | {'Units':<15} | {'Revenue':<15} | {'Top buyer':<15}"
)
width = len(header)
print("=" * width)
print(f"{'PRODUCT SUMMARIES':^{width}}")
print("=" * width)
print(header)
print("-" * width)
for prod, s in product_summary.items():
    print(
        f"{prod:<12} | {s['txns']:<12} | {s['units']:<15} | {s['revenue']:<15} | {s['top_buyer']:<15}"
    )
# best_sellers and top_revenue generated by ChatGPT
best_units = max(v["units"] for v in product_totals.values())
best_sellers = [
    p for p, v in product_totals.items()
    if v["units"] == best_units
]

best_revenue = max(v["revenue"] for v in product_totals.values())
top_revenue = [
    p for p, v in product_totals.items()
    if v["revenue"] == best_revenue
]

print(f"Best-Selling Product (in units):  {" , ".join(best_sellers)} with {best_units} units")
print(f"Highest-Revenue Product: {" , ".join(top_revenue)} with {best_revenue:.2f}")
print("=" * width)

# ORDERED RESULTS: Highest to lowest ranking
ranked_trxns = sorted(distinct_trxns, key=lambda t: (-t['amount'], t['customer']))
header = (
     f"{'Rank':<12} | {'Customer':<12} | {'Product':<12} | {'Qty':<15} | {'Unit Price':<15} | {'Total Amount':<15}"
)
width = len(header)
print("=" * width)
print(f"{'ORDERED RESULTS (from highest to lowest spending)':^{width}}")
print("=" * width)
print(header)
print("-" * width)
for rank, t in enumerate(ranked_trxns, start=1):
    print(
        f"{rank:<12} | {t['customer']:<12} | {t['product']:<12} | {t['quantity']:<15} | {PRODUCTS[t["product"]]:<15} | {t['amount']:<15}"
    )
print("-" * width)
print(f"Total: {sum(t['amount'] for t in ranked_trxns):.2f}")
print("=" * width)


total_txns = len(prod_trxns)
total_units = sum(t["quantity"] for t in prod_trxns)
total_revenue = round(sum(t["amount"] for t in prod_trxns), 2)
average_trxn_value = round(total_revenue / total_txns, 2) if total_txns else 0
print("=" * 100)
print(f"{'CONSOLIDATED TRANSACTION REPORT':^100}")
print("-" * 100)
top_pair = max(purchase_records, key=lambda r: r['total'])
print(f"Total Revenue: {total_revenue}")
print(f"Total Transactions: {total_txns}")
print(f"Total Units Sold: {total_units}")
print(f"Average Transaction Value: {average_trxn_value}")
print(f"Top Customer-Product Pair: {top_pair['customer']} --> {top_pair['product']} with: {top_pair['units']}, and {top_pair['total']:.2f}")
print("=" * 100)
print()


GENERATED TRANSACTION RECORDS
----------------------------------------------------------------------------------------------------
Customer: Bacor_C1, Product: CALCULATOR, Quantity 5, Amount: 6000.0
Customer: Bacor_C2, Product: PEN, Quantity 1, Amount: 10.0
Customer: Bacor_C3, Product: HEADPHONES, Quantity 2, Amount: 200.0
Customer: Bacor_C4, Product: NOTEBOOK, Quantity 3, Amount: 150.0
Customer: Bacor_C5, Product: HEADPHONES, Quantity 4, Amount: 400.0
Customer: Bacor_C1, Product: CALCULATOR, Quantity 5, Amount: 6000.0
Customer: Bacor_C2, Product: USB, Quantity None, Amount: None
                 ORGANIZED TRANSACTION RECORDS                  
Customer     | Product      | Qty             | Unit Price      
----------------------------------------------------------------
Bacor_C1     | CALCULATOR   | 5               | 1200.0         
Bacor_C1     | CALCULATOR   | 5               | 1200.0         
Bacor_C2     | PEN          | 1               | 10.0           
Bacor_C2     | USB        